# Create Research Council of Norway Awards

Creates OpenAlex award rows from the Research Council of Norway's official Project Bank.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/research_council_norway_to_s3.py` to fetch the official Project Bank JSON route, write parquet, and upload it.
- Contractor has prepared the script, notebook, registry entry, tracker row, and local QA, but does not have S3/Databricks access.
- The downloader writes parquet columns as strings per `plans/awards/how-to-add-a-funder.md`; this notebook does type conversion with `TRY_CAST` / `TRY_TO_DATE`.

**Data source:** `https://prosjektbanken.forskningsradet.no/en/explore/projects` via the official Next.js JSON route `/_next/data/{build_id}/en/explore/projects.json` with `Kilde=FORISS`.  
**S3 location:** `s3a://openalex-ingest/awards/research_council_norway/research_council_norway_projects.parquet`  
**Local full-source validation on 2026-05-27:** 44,777 official `FORISS` project rows advertised by the Project Bank JSON route; 1989-2026 start-year range; NOK 211.97B total funding.  
**OpenAlex funder:** `F4320323299` - Research Council of Norway / Norges Forskningsrad (OpenAlex display name has Norwegian spelling), country `NO`; no ROR/DOI exposed by the public OpenAlex API at Step 0.  
**Provenance:** `research_council_norway_project_bank` (OpenAlex production preflight returned 0 rows).  
**Priority:** 130.

**Schema choices:**
- One row per official Project Bank `FORISS` project record.
- Stable `funder_award_id = rcn-foriss-{project_id}` from source + project id.
- `amount = total_funding` and `currency = NOK`; the Project Bank documentation describes these as Research Council funding allocations, and the UI renders them as Norwegian kroner.
- `funding_type = 'research'`; `funder_scheme = current_activity_code` from the Project Bank activity/program field.
- `lead_investigator` is populated from `lead_name` plus the first source organization hierarchy; ORCID is not published and remains NULL.
- EU and SkatteFUNN rows are deliberately excluded because they are separate Project Bank views, not direct Research Council awards.

**Contractor handoff:** Contractor has no S3/Databricks access; admin must upload the parquet, run this notebook, inspect the Step 6 verification cells, then run `CreateAwards.ipynb`.


## Step 1: Create Raw Table


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.research_council_norway_raw
USING delta
AS
SELECT
    *,
    current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/research_council_norway/research_council_norway_projects.parquet`;


In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-27 found 44,777 FORISS project records.
SELECT COUNT(*) AS total_research_council_norway_projects
FROM openalex.awards.research_council_norway_raw;


## Step 1.5: Inspect Raw Data, Money Fields, and Key Uniqueness

Expected source columns: `source_record_id`, `project_id`, `source`, `funder_award_id`, `display_name`, `teaser`, `pop_sci_description`, `project_summary`, `lead_name`, `lead_given_name`, `lead_family_name`, `primary_organization`, `organizations_json`, `geographies_json`, `disciplines_json`, `current_activity_code`, `current_activity_year`, `current_activity_json`, `years_active_json`, `start_year`, `end_year`, `start_date`, `end_date`, `total_funding`, `currency`, `landing_page_url`, `source_list_url`, `downloaded_at`.

All columns are STRING in the parquet. The transform below casts amount/date/year fields explicitly.


In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.research_council_norway_raw;


In [ ]:
%sql
-- Sample raw Project Bank data.
SELECT
    funder_award_id,
    project_id,
    display_name,
    lead_name,
    primary_organization,
    total_funding,
    currency,
    start_year,
    end_year,
    current_activity_code,
    landing_page_url
FROM openalex.awards.research_council_norway_raw
LIMIT 10;


In [ ]:
%sql
-- Money-field scan per the runbook. Do not use DESCRIBE as a subquery.
SELECT column_name FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'research_council_norway_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|grant_offer|awarded|valeur|monto|importe|montant|betrag|valor|importo|kwota|belopp';


In [ ]:
%sql
SELECT
    MIN(TRY_CAST(total_funding AS DOUBLE)) AS min_total_funding,
    MAX(TRY_CAST(total_funding AS DOUBLE)) AS max_total_funding,
    ROUND(AVG(TRY_CAST(total_funding AS DOUBLE)), 2) AS avg_total_funding,
    COUNT(total_funding) AS non_null_total_funding,
    COUNT(*) AS total_rows,
    ROUND(try_divide(COUNT(total_funding), COUNT(*)) * 100.0, 1) AS pct_total_funding,
    collect_set(currency) AS currencies
FROM openalex.awards.research_council_norway_raw;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(display_name) AS has_title,
    COUNT(project_summary) AS has_project_summary,
    COUNT(pop_sci_description) AS has_pop_sci_description,
    COUNT(lead_name) AS has_lead_name,
    COUNT(primary_organization) AS has_primary_organization,
    COUNT(start_date) AS has_start_date,
    COUNT(end_date) AS has_end_date,
    COUNT(current_activity_code) AS has_current_activity_code,
    MIN(TRY_CAST(start_year AS INT)) AS min_start_year,
    MAX(TRY_CAST(start_year AS INT)) AS max_start_year,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT project_id) AS distinct_project_ids
FROM openalex.awards.research_council_norway_raw;


## Step 1.6: Fail-Fast - Verify Research Council of Norway Funder Exists

The Step 2 transform joins to `openalex.common.funder`. This must return exactly one row.


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320323299;


## Step 2: Transform to OpenAlex Awards Schema


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.research_council_norway_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320323299
),
raw_prepped AS (
    SELECT
        *,
        TRY_CAST(total_funding AS DOUBLE) AS amount_double,
        CASE
            WHEN YEAR(TRY_TO_DATE(start_date, 'yyyy-MM-dd')) BETWEEN 1800 AND 2100
                THEN TRY_TO_DATE(start_date, 'yyyy-MM-dd')
            ELSE NULL
        END AS start_date_cast,
        CASE
            WHEN YEAR(TRY_TO_DATE(end_date, 'yyyy-MM-dd')) BETWEEN 1800 AND 2100
                THEN TRY_TO_DATE(end_date, 'yyyy-MM-dd')
            ELSE NULL
        END AS end_date_cast,
        CASE WHEN TRY_CAST(start_year AS INT) BETWEEN 1800 AND 2100 THEN TRY_CAST(start_year AS INT) END AS start_year_int,
        CASE WHEN TRY_CAST(end_year AS INT) BETWEEN 1800 AND 2100 THEN TRY_CAST(end_year AS INT) END AS end_year_int,
        NULLIF(TRIM(lead_given_name), '') AS leader_given_name,
        NULLIF(TRIM(lead_family_name), '') AS leader_family_name,
        NULLIF(TRIM(lead_name), '') AS leader_full_name,
        NULLIF(TRIM(primary_organization), '') AS leader_affiliation_name,
        CONCAT_WS('\n\n',
            CASE WHEN NULLIF(TRIM(pop_sci_description), '') IS NOT NULL THEN CONCAT('Popular science description: ', TRIM(pop_sci_description)) END,
            CASE WHEN NULLIF(TRIM(project_summary), '') IS NOT NULL THEN CONCAT('Project summary: ', TRIM(project_summary)) END,
            CASE WHEN NULLIF(TRIM(teaser), '') IS NOT NULL THEN CONCAT('Teaser: ', TRIM(teaser)) END,
            CASE WHEN NULLIF(TRIM(disciplines_json), '') IS NOT NULL THEN CONCAT('Disciplines: ', TRIM(disciplines_json)) END,
            CASE WHEN NULLIF(TRIM(geographies_json), '') IS NOT NULL THEN CONCAT('Geographies: ', TRIM(geographies_json)) END
        ) AS award_description
    FROM openalex.awards.research_council_norway_raw
)
SELECT
    abs(xxhash64(CONCAT('4320323299:', LOWER(TRIM(rp.funder_award_id))))) % 9000000000 AS id,
    rp.display_name,
    NULLIF(rp.award_description, '') AS description,
    4320323299 AS funder_id,
    rp.funder_award_id,
    rp.amount_double AS amount,
    CASE WHEN rp.amount_double IS NOT NULL THEN 'NOK' ELSE NULL END AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'research' AS funding_type,
    NULLIF(TRIM(rp.current_activity_code), '') AS funder_scheme,
    'research_council_norway_project_bank' AS provenance,
    rp.start_date_cast AS start_date,
    rp.end_date_cast AS end_date,
    rp.start_year_int AS start_year,
    rp.end_year_int AS end_year,
    CASE
        WHEN rp.leader_full_name IS NOT NULL OR rp.leader_affiliation_name IS NOT NULL THEN
            struct(
                rp.leader_given_name AS given_name,
                COALESCE(rp.leader_family_name, rp.leader_full_name) AS family_name,
                CAST(NULL AS STRING) AS orcid,
                rp.start_date_cast AS role_start,
                struct(
                    rp.leader_affiliation_name AS name,
                    CASE WHEN rp.leader_affiliation_name IS NOT NULL THEN 'NO' ELSE NULL END AS country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
                ) AS affiliation
            )
        ELSE NULL
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    rp.landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT('4320323299:', LOWER(TRIM(rp.funder_award_id))))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM raw_prepped rp
JOIN funder_resolved f ON f.funder_id = 4320323299
WHERE rp.funder_award_id IS NOT NULL
  AND rp.display_name IS NOT NULL;


## Step 3: Insert into `openalex_awards_raw` at Priority 130


In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'research_council_norway_project_bank' AND priority = 130;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    130 AS priority
FROM openalex.awards.research_council_norway_awards;


## Step 6: Verification

Grant pattern with populated NOK amount/currency from the official Project Bank `total_funding` field. Amount coverage should be near 100%; investigate if it drops materially below the local validation run.


In [ ]:
%sql
SELECT COUNT(*) AS total_research_council_norway_awards
FROM openalex.awards.research_council_norway_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.research_council_norway_awards;


In [ ]:
%sql
-- Duplicate funder_award_id must be zero; duplicate ids must also be zero.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT id) AS distinct_openalex_award_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS duplicate_funder_award_ids,
    COUNT(*) - COUNT(DISTINCT id) AS duplicate_openalex_ids
FROM openalex.awards.research_council_norway_awards;


In [ ]:
%sql
-- Data completeness check. Percentages use try_divide to avoid divide-by-zero on empty tables.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS has_positive_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(end_date) AS has_end_date,
    SUM(CASE WHEN lead_investigator.family_name IS NOT NULL THEN 1 ELSE 0 END) AS has_lead_name,
    SUM(CASE WHEN lead_investigator.affiliation.name IS NOT NULL THEN 1 ELSE 0 END) AS has_lead_affiliation,
    COUNT(funder_scheme) AS has_funder_scheme,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description,
    ROUND(try_divide(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_positive_amount,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) AS pct_start_date,
    ROUND(try_divide(SUM(CASE WHEN lead_investigator.family_name IS NOT NULL THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_lead_name,
    collect_set(currency) AS currencies
FROM openalex.awards.research_council_norway_awards;


In [ ]:
%sql
-- Amount and currency coverage. This is not waived: Project Bank exposes total_funding.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    ROUND(AVG(amount), 2) AS avg_amount,
    percentile_approx(amount, 0.5) AS median_amount
FROM openalex.awards.research_council_norway_awards;


In [ ]:
%sql
SELECT start_year, COUNT(*) AS awards, ROUND(SUM(amount), 0) AS nok_total
FROM openalex.awards.research_council_norway_awards
GROUP BY start_year
ORDER BY start_year DESC;


In [ ]:
%sql
SELECT funder_scheme, COUNT(*) AS awards, ROUND(SUM(amount), 0) AS nok_total
FROM openalex.awards.research_council_norway_awards
GROUP BY funder_scheme
ORDER BY awards DESC
LIMIT 40;


In [ ]:
%sql
SELECT
    id,
    SUBSTRING(display_name, 1, 90) AS title,
    funder_award_id,
    amount,
    currency,
    start_date,
    end_date,
    lead_investigator.family_name AS lead_name,
    SUBSTRING(lead_investigator.affiliation.name, 1, 70) AS affiliation,
    landing_page_url
FROM openalex.awards.research_council_norway_awards
ORDER BY start_year DESC, id
LIMIT 20;


In [ ]:
%sql
-- Confirm the priority insert landed as expected.
SELECT priority, provenance, COUNT(*) AS rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'research_council_norway_project_bank'
GROUP BY priority, provenance;


## Handoff Notes

- Contractor-complete state only: the script, notebook, registry entry, tracker row, and local validation are ready.
- Do not add job YAML until an admin has uploaded the parquet, run this notebook on Databricks, inspected Step 6, and run `CreateAwards.ipynb` successfully.
- Contractor has no S3/Databricks access; admin must run upload + Databricks notebook + QA.
